In [1]:
!pip install mlflow torch==2.7.0 petastorm s3fs "pyarrow==10.0.1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.0/68.0 kB 633.6 kB/s eta 0:00:00a 0:00:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.9/73.9 kB 775.6 kB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 865.2/865.2 MB 3.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 13.5 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 393.1/393.1 MB 6.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 28.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 22.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 897.7/897.7 kB 10.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 571.0/571.0 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.2/200.2 MB 13.4 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 23.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━

In [2]:
!pip install --upgrade --force-reinstall "s3fs>=2023.9.0" "fsspec>=2023.9.0"

  Using cached s3fs-2025.5.1-py3-none-any.whl.metadata (1.9 kB)
  Using cached fsspec-2025.5.1-py3-none-any.whl.metadata (11 kB)
  Using cached aiobotocore-2.23.0-py3-none-any.whl.metadata (24 kB)
  Using cached aiohttp-3.12.13-cp311-cp311-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (7.6 kB)
  Using cached aioitertools-0.12.0-py3-none-any.whl.metadata (3.8 kB)
  Using cached botocore-1.38.27-py3-none-any.whl.metadata (5.7 kB)
  Using cached jmespath-1.0.1-py3-none-any.whl.metadata (7.6 kB)
  Using cached multidict-6.6.3-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (5.3 kB)
  Using cached wrapt-1.17.2-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_64.manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.4 kB)
  Using cached aiohappyeyeballs-2.6.1-py3-none-any.whl.metadata (5.9 kB)
  Using cached aiosignal-1.4.0-py3-none-any.whl.metadata (3.7 kB)
  Using cached frozenlist-1.7.0-cp311-cp311-manylinux_2_5_x86_64.manylinux1_x86_

In [3]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import input_file_name, regexp_extract, lit

# --- 1. Configure Spark Session ---
spark = (
    SparkSession.builder.appName("IoT_Botnet_Transformation")
    .master("local[*]")
    .config("spark.driver.memory", "4g")
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000")
    .config("spark.hadoop.fs.s3a.access.key", "minio_access_key")
    .config("spark.hadoop.fs.s3a.secret.key", "minio_secret_key")
    .config("spark.hadoop.fs.s3a.path.style.access", True)
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.jars.packages", "io.delta:delta-spark_2.12:3.2.0,org.apache.hadoop:hadoop-aws:3.3.4")
    .getOrCreate()
)
print("Spark session created successfully!")

Spark session created successfully!


In [4]:
# --- 2. Read and Transform Data ---
raw_path = "s3a://iot-time-series/raw_data/*/*.csv"

# Read all CSVs. Using input_file_name() lets us extract info from the path.
df = spark.read.option("header", "true").option("inferSchema", "true").csv(raw_path)

# Extract device name and attack type from the file path
# The path looks like: s3a://raw/Danmini_Doorbell/benign_traffic.csv
path_regex = r"s3a:\/\/iot-time-series\/(.*?)/(.*?)\/(.*?.csv)"
df = df.withColumn("filePath", input_file_name())
df = df.withColumn("device", regexp_extract("filePath", path_regex, 2))
df = df.withColumn("label", regexp_extract("filePath", path_regex, 3))
df = df.withColumn("label", regexp_extract("label", r"(.*?)_.*", 1)) # Extract attack name, e.g., 'benign' or 'gafgyt'
df = df.drop("filePath")

df.printSchema()
df.select("device", "label").show(5)

root
 |-- MI_dir_L5_weight: double (nullable = true)
 |-- MI_dir_L5_mean: double (nullable = true)
 |-- MI_dir_L5_variance: double (nullable = true)
 |-- MI_dir_L3_weight: double (nullable = true)
 |-- MI_dir_L3_mean: double (nullable = true)
 |-- MI_dir_L3_variance: double (nullable = true)
 |-- MI_dir_L1_weight: double (nullable = true)
 |-- MI_dir_L1_mean: double (nullable = true)
 |-- MI_dir_L1_variance: double (nullable = true)
 |-- MI_dir_L0.1_weight: double (nullable = true)
 |-- MI_dir_L0.1_mean: double (nullable = true)
 |-- MI_dir_L0.1_variance: double (nullable = true)
 |-- MI_dir_L0.01_weight: double (nullable = true)
 |-- MI_dir_L0.01_mean: double (nullable = true)
 |-- MI_dir_L0.01_variance: double (nullable = true)
 |-- H_L5_weight: double (nullable = true)
 |-- H_L5_mean: double (nullable = true)
 |-- H_L5_variance: double (nullable = true)
 |-- H_L3_weight: double (nullable = true)
 |-- H_L3_mean: double (nullable = true)
 |-- H_L3_variance: double (nullable = true)
 |

In [9]:
# --- 3. Write to Delta Lake format ---
processed_path = "s3a://processed/iot_botnet_data"

# Save the DataFrame as a Delta table, overwriting if it already exists
df.write.format("delta").mode("overwrite").save(processed_path)

print(f"Successfully wrote data to Delta table at: {processed_path}")

Successfully wrote data to Delta table at: s3a://processed/iot_botnet_data


In [10]:
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, VectorAssembler, StandardScaler
from pyspark.sql.functions import udf
from pyspark.sql.types import ArrayType, DoubleType

# --- 1. Load the Cleaned Delta Table ---
print("Loading data from Delta Lake...")
processed_path = "s3a://processed/iot_botnet_data"
delta_df = spark.read.format("delta").load(processed_path)

# --- 2. Sanitize Column Names ---
print("Sanitizing column names...")
sanitized_df = delta_df
for col_name in delta_df.columns:
    if "." in col_name:
        new_col_name = col_name.replace(".", "_")
        sanitized_df = sanitized_df.withColumnRenamed(col_name, new_col_name)

# --- 3. Create a Feature Engineering Pipeline in Spark ---
print("Building and applying Spark ML preprocessing pipeline...")
feature_cols = [c for c, t in sanitized_df.dtypes if t in ('int', 'double') and c != 'label']
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")
label_indexer = StringIndexer(inputCol="label", outputCol="indexedLabel")
pipeline = Pipeline(stages=[assembler, scaler, label_indexer])
preprocessor_model = pipeline.fit(sanitized_df)
training_ready_df = preprocessor_model.transform(sanitized_df)

# --- 4. FIX: Convert Vector to Array using a UDF ---
print("Converting feature vector to a simple array...")
# Define the function to convert a vector to a list
def vector_to_array(v):
    return v.toArray().tolist()

# Register the function as a UDF
vector_to_array_udf = udf(vector_to_array, ArrayType(DoubleType()))

# Apply the UDF to create a new column
final_df_with_vector = training_ready_df.withColumn("features_flat", vector_to_array_udf("scaledFeatures"))

# --- 5. Save the Final Training-Ready Data ---
# Select only the columns needed for training with the corrected format
final_df = final_df_with_vector.select("features_flat", "indexedLabel")

# Save to a new location in Delta format
feature_store_path = "s3a://processed/training_ready_features"
final_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(feature_store_path)

print(f"Successfully saved training-ready data with flattened features to: {feature_store_path}")

Loading data from Delta Lake...
Sanitizing column names...
Building and applying Spark ML preprocessing pipeline...
Converting feature vector to a simple array...
Successfully saved training-ready data with flattened features to: s3a://processed/training_ready_features


In [2]:
import mlflow
import mlflow.pytorch
import torch
import torch.nn as nn
from petastorm import make_reader
from petastorm.pytorch import DataLoader

# --- 1. Define device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")


# --- 2. Model Architecture ---
class MLPDetector(nn.Module):
    def __init__(self, input_size, hidden_size_1, hidden_size_2, output_size, dropout_rate=0.3):
        super(MLPDetector, self).__init__()
        self.layer_1 = nn.Linear(input_size, hidden_size_1)
        self.relu_1 = nn.ReLU()
        self.dropout_1 = nn.Dropout(dropout_rate)
        self.layer_2 = nn.Linear(hidden_size_1, hidden_size_2)
        self.relu_2 = nn.ReLU()
        self.dropout_2 = nn.Dropout(dropout_rate)
        self.output_layer = nn.Linear(hidden_size_2, output_size)

    def forward(self, x):
        x = self.relu_1(self.dropout_1(self.layer_1(x)))
        x = self.relu_2(self.dropout_2(self.layer_2(x)))
        x = self.output_layer(x)
        return x

# --- 3. MLflow Experiment Tracking ---
mlflow.set_tracking_uri("http://mlflow_server:5000")
mlflow.set_experiment("iot_botnet_detection_pytorch_batch")

with mlflow.start_run() as run:
    # --- Hyperparameters ---
    LEARNING_RATE = 0.001
    EPOCHS = 5
    INPUT_SIZE = 115

    # --- Move model to device ---
    model = MLPDetector(input_size=INPUT_SIZE, hidden_size_1=128, hidden_size_2=64, output_size=1, dropout_rate=0.3).to(device)

    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

    mlflow.log_params({"learning_rate": LEARNING_RATE, "epochs": EPOCHS, "batch_size": 256, "architecture": "MLP-128-64"})

    # --- Training Loop ---
    print(f"Beginning training for {EPOCHS} epochs on {device}...")
    feature_store_path = "s3a://processed/training_ready_features"
    storage_options = { "key": "minio_access_key", "secret": "minio_secret_key", "endpoint_url": "http://minio:9000", "use_ssl": False }

    with DataLoader(make_reader(
        dataset_url=feature_store_path,
        reader_pool_type='thread',
        storage_options=storage_options
    )) as train_loader:
        for epoch in range(EPOCHS):
            model.train()
            total_loss = 0
            # 1. Initialize batch counter
            num_batches = 0

            for batch in train_loader:
                # Move data to device
                features = batch['features_flat'].float().to(device)
                labels = batch['indexedLabel'].float().unsqueeze(1).to(device)

                outputs = model(features)
                loss = criterion(outputs, labels)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item()
                # 2. Increment batch counter
                num_batches += 1

            # 3. Use the counter to calculate average loss
            if num_batches > 0:
                avg_loss = total_loss / num_batches
                print(f"Epoch [{epoch+1}/{EPOCHS}], Average Loss: {avg_loss:.4f}")
                mlflow.log_metric("training_loss", avg_loss, step=epoch)

    # # --- Training Loop ---
    # print(f"Beginning training for {EPOCHS} epochs on {device}...")
    # feature_store_path = "s3a://processed/training_ready_features"
    # storage_options = { "key": "minio_access_key", "secret": "minio_secret_key", "endpoint_url": "http://minio:9000", "use_ssl": False }

    # with DataLoader(make_reader(dataset_url=feature_store_path, reader_pool_type='thread', storage_options=storage_options)) as train_loader:
    #     for epoch in range(EPOCHS):
    #         model.train()
    #         total_loss = 0
    #         for batch in train_loader:
    #             # --- Move data to device ---
    #             features = batch['features_flat'].float().to(device)
    #             labels = batch['indexedLabel'].float().unsqueeze(1).to(device)

    #             outputs = model(features)
    #             loss = criterion(outputs, labels)

    #             optimizer.zero_grad()
    #             loss.backward()
    #             optimizer.step()
    #             total_loss += loss.item()

    #         avg_loss = total_loss / len(train_loader)
    #         print(f"Epoch [{epoch+1}/{EPOCHS}], Average Loss: {avg_loss:.4f}")
    #         mlflow.log_metric("training_loss", avg_loss, step=epoch)

    # --- Log the Model ---
    print("Logging model to MLflow...")
    mlflow.pytorch.log_model(model, "model")

print("\nPyTorch batch training and tracking complete.")

Using device: cuda
Beginning training for 5 epochs on cuda...


/opt/conda/lib/python3.11/site-packages/petastorm/etl/dataset_metadata.py:402: FutureWarning: Specifying the 'metadata_nthreads' argument is deprecated as of pyarrow 8.0.0, and the argument will be removed in a future version
  dataset = pq.ParquetDataset(path_or_paths, filesystem=fs, validate_schema=False, metadata_nthreads=10)
/opt/conda/lib/python3.11/site-packages/petastorm/etl/dataset_metadata.py:362: FutureWarning: 'ParquetDataset.common_metadata' attribute is deprecated as of pyarrow 5.0.0 and will be removed in a future version.
  if not dataset.common_metadata:
/opt/conda/lib/python3.11/site-packages/petastorm/reader.py:158: UserWarning: Currently make_reader supports reading only Petastorm datasets. To read from a non-Petastorm Parquet store use make_batch_reader
  warnings.warn('Currently make_reader supports reading only Petastorm datasets. '
/opt/conda/lib/python3.11/site-packages/petastorm/reader.py:420: FutureWarning: Specifying the 'metadata_nthreads' argument is deprec

Epoch [1/5], Average Loss: 0.0000


/opt/conda/lib/python3.11/site-packages/petastorm/py_dict_reader_worker.py:146: FutureWarning: 'ParquetDataset.fs' attribute is deprecated as of pyarrow 5.0.0 and will be removed in a future version. Specify 'use_legacy_dataset=False' while constructing the ParquetDataset, and then use the '.filesystem' attribute instead.
  parquet_file = ParquetFile(self._dataset.fs.open(piece.path))
/opt/conda/lib/python3.11/site-packages/petastorm/py_dict_reader_worker.py:182: FutureWarning: 'ParquetDataset.partitions' attribute is deprecated as of pyarrow 5.0.0 and will be removed in a future version. Specify 'use_legacy_dataset=False' while constructing the ParquetDataset, and then use the '.partitioning' attribute instead.
  partitions = self._dataset.partitions
/opt/conda/lib/python3.11/site-packages/petastorm/py_dict_reader_worker.py:267: FutureWarning: 'ParquetDataset.partitions' attribute is deprecated as of pyarrow 5.0.0 and will be removed in a future version. Specify 'use_legacy_dataset=Fa

Epoch [2/5], Average Loss: 0.0000


...

Epoch [4/5], Average Loss: 0.0000


/opt/conda/lib/python3.11/site-packages/petastorm/py_dict_reader_worker.py:146: FutureWarning: 'ParquetDataset.fs' attribute is deprecated as of pyarrow 5.0.0 and will be removed in a future version. Specify 'use_legacy_dataset=False' while constructing the ParquetDataset, and then use the '.filesystem' attribute instead.
  parquet_file = ParquetFile(self._dataset.fs.open(piece.path))
/opt/conda/lib/python3.11/site-packages/petastorm/py_dict_reader_worker.py:182: FutureWarning: 'ParquetDataset.partitions' attribute is deprecated as of pyarrow 5.0.0 and will be removed in a future version. Specify 'use_legacy_dataset=False' while constructing the ParquetDataset, and then use the '.partitioning' attribute instead.
  partitions = self._dataset.partitions
/opt/conda/lib/python3.11/site-packages/petastorm/py_dict_reader_worker.py:267: FutureWarning: 'ParquetDataset.partitions' attribute is deprecated as of pyarrow 5.0.0 and will be removed in a future version. Specify 'use_legacy_dataset=Fa

Epoch [5/5], Average Loss: 0.0000
Logging model to MLflow...


2025/07/10 14:32:18 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


🏃 View run funny-wren-202 at: http://mlflow_server:5000/#/experiments/985621612400213845/runs/aa01892edfbd4a70be8b20ffcf52d32a
🧪 View experiment at: http://mlflow_server:5000/#/experiments/985621612400213845

PyTorch batch training and tracking complete.
